# AlphaGenome — Variant Analysis Tutorial

**DII Bioinformatics Club · Luxembourg Institute of Health**

Welcome! In this tutorial we'll walk through the core Python workflow for
predicting the effect of a genetic variant using [AlphaGenome](https://www.nature.com/articles/s41586-025-10014-0),
Google DeepMind's new foundation model for genomics.

**No coding experience is required** — we'll explain every line as we go. The
goal is for you to understand *what the code is doing* and *why*, not to
memorise the syntax.

---

### What we'll cover (~30 minutes)

| Step | What we'll do | Time |
|------|---------------|------|
| 1 | Install & import the SDK | 2 min |
| 2 | Connect to the model | 2 min |
| 3 | Define a genetic variant | 5 min |
| 4 | Predict REF vs ALT allele effects | 8 min |
| 5 | Score the variant across all tissues | 8 min |
| 6 | Interpret the results | 5 min |

---

### Quick biology reminder

A **variant** (or SNV — single nucleotide variant) is a single DNA base that
differs between individuals. For example, at position 36,201,698 on chromosome
22, some people have an **A** (the reference allele) while others have a **C**
(the alternative allele).

AlphaGenome can predict how this single base change affects gene expression,
chromatin accessibility, splicing, and more — across different tissues —
entirely from the DNA sequence.

---
## Step 1 · Install AlphaGenome

This cell installs the `alphagenome` Python package. It only needs to run once
per session. The `clear_output()` at the end just tidies up the install log.

In [ ]:
# Install the AlphaGenome SDK (only needed once per session)
from IPython.display import clear_output
! pip install alphagenome
clear_output()
print("✅ AlphaGenome installed successfully!")

---
## Step 2 · Import libraries and connect to the model

We import the modules we'll need. Here's what each one does:

| Import | Purpose |
|--------|---------|
| `genome` | Define genomic coordinates and variants |
| `dna_client` | Connect to the AlphaGenome API |
| `variant_scorers` | Score the effect of variants |
| `gene_annotation` | Look up gene locations from GENCODE |
| `transcript_utils` | Extract transcript structures for plots |
| `plot_components` | Visualise predictions as genome browser–style tracks |

In [ ]:
# Core SDK modules
from alphagenome.data import genome
from alphagenome.data import gene_annotation
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components

# Standard libraries
import matplotlib.pyplot as plt
import pandas as pd

### Connect to the model

AlphaGenome runs on Google's servers — we access it via an **API key** (like a
password). The line below creates a model object that we'll use for all our
predictions.

> **Colab users:** store your key in the Colab "Secrets" panel (🔑 icon in the
> left sidebar) with the name `ALPHAGENOME_API_KEY`. The helper function below
> will retrieve it automatically.
>
> **Everyone else:** you can pass the key directly:
> `dna_client.create("your-key-here")`

In [ ]:
from alphagenome import colab_utils

# Create the model connection
dna_model = dna_client.create(colab_utils.get_api_key())
print("✅ Connected to AlphaGenome!")

Let's quickly check which **output types** (i.e. assay categories) the model
can predict. These correspond to real experimental techniques like RNA-seq,
ATAC-seq, DNase-seq, etc.

In [ ]:
# List all available output types
for output in dna_client.OutputType:
    print(f"  {output.name}")

---
## Step 3 · Load gene annotations

Before we can overlay gene structures on our plots, we need a reference file
that tells us *where every gene is* in the human genome. This is called a
**GTF file** (Gene Transfer Format) from the [GENCODE](https://www.gencodegenes.org/) project.

The first time you run this cell it downloads ~100 MB from Google Cloud Storage.
After that it's cached in memory, so subsequent calls are instant.

We filter to **MANE Select transcripts** — these are the single "best"
curated transcript per gene, agreed on by NCBI and Ensembl. This keeps our
plots clean and easy to read.

In [ ]:
# Download gene annotations (human genome assembly GRCh38 / hg38)
# ⏱️ First run downloads ~100 MB — takes ~30 seconds
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

# Keep only protein-coding genes, one MANE Select transcript per gene
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_mane_select_transcript(gtf_transcripts)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

print(f"✅ Loaded {len(gtf_transcripts):,} MANE Select transcripts")

---
## Step 4 · Define a genetic variant

Now for the interesting part! We'll define a specific variant and ask
AlphaGenome to predict its effect.

### The variant we're studying

| Property | Value | Meaning |
|----------|-------|---------|
| Chromosome | chr22 | Chromosome 22 |
| Position | 36,201,698 | The specific base (1-based, VCF-style) |
| REF allele | A | The reference genome has an A here |
| ALT allele | C | We want to test what happens with a C instead |
| Known effect | eQTL in colon | This variant is known to affect gene expression in colon tissue |

This is a real variant from human population studies — it's been identified as
an **eQTL** (expression quantitative trait locus), meaning it's statistically
associated with changes in gene expression in colon tissue.

In [ ]:
# Define the variant using VCF-style coordinates
variant = genome.Variant(
    chromosome='chr22',
    position=36201698,       # 1-based coordinate
    reference_bases='A',     # Reference allele
    alternate_bases='C',     # Alternative allele we want to test
)

print(f"Variant: {variant}")

### Create the prediction window

AlphaGenome needs a fixed-length DNA sequence as input. We'll centre a **1 Mb
(1 million base pair)** window on our variant — this gives the model enough
surrounding context to capture long-range regulatory effects.

Think of it like zooming out on a genome browser: we need to show the model
enough of the neighbourhood around the variant for it to understand the
regulatory landscape.

In [ ]:
# Create a 1 Mb interval centred on the variant
interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

print(f"Prediction window: {interval}")
print(f"Window size: {interval.width:,} bp ({interval.width / 1_000_000:.1f} Mb)")

---
## Step 5 · Predict the variant's effect on gene expression

This is the core analysis. We ask AlphaGenome to make **two predictions**:

1. **REF prediction** — what the model predicts with the reference allele (A)
2. **ALT prediction** — what the model predicts with the alternative allele (C)

By comparing REF vs ALT, we can see exactly how this single base change affects
the predicted RNA-seq signal.

We'll look at **colon transverse** tissue (`UBERON:0001157`) because this is
where the variant is known to have an effect.

> **What's an ontology term?** It's a standardised identifier for a tissue or
> cell type. `UBERON:0001157` is the universal code for "transverse colon" —
> it works like a barcode that means the same thing across all databases.

In [ ]:
# Predict RNA-seq for REF and ALT alleles in colon tissue
# ⏱️ This takes ~30-60 seconds (two API calls: one for REF, one for ALT)
variant_output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs=[dna_client.OutputType.RNA_SEQ],
    ontology_terms=['UBERON:0001157'],  # Colon - Transverse
)

print("✅ Predictions complete!")
print(f"   REF tracks shape: {variant_output.reference.rna_seq.values.shape}")
print(f"   ALT tracks shape: {variant_output.alternate.rna_seq.values.shape}")

### Visualise REF vs ALT

Now let's plot the REF (grey) and ALT (red) predictions overlaid on top of each
other. We'll also show:
- **Gene transcripts** at the top (boxes = exons, lines = introns, arrows = strand)
- A **vertical line** marking the variant's position

We zoom in to a 32 kb window around the variant so you can clearly see the
difference.

In [ ]:
# Extract gene transcripts for this region
transcripts = transcript_extractor.extract(interval)

# Plot REF vs ALT overlaid, zoomed to 32 kb around the variant
plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts),
        plot_components.OverlaidTracks(
            tdata={
                'REF': variant_output.reference.rna_seq,
                'ALT': variant_output.alternate.rna_seq,
            },
            colors={'REF': 'dimgrey', 'ALT': 'red'},
        ),
    ],
    interval=variant_output.reference.rna_seq.interval.resize(2**15),
    # Mark the variant position with a vertical line
    annotations=[plot_components.VariantAnnotation([variant], alpha=0.8)],
)
plt.show()

### 🔍 What to look for in the plot

Look at the **negative strand tracks** (the lower panels — check the y-axis
scale to find the ones with meaningful signal).

You should see two clear effects of the A→C variant:

1. **Lower expression** — the red (ALT) signal is generally lower than the
   grey (REF) signal across the gene body
2. **Exon skipping** — at one of the exons, the red signal drops to near zero
   while the grey signal remains high, suggesting the ALT allele causes an exon
   to be skipped

The affected gene is **APOL4** (Apolipoprotein L4) on the negative strand.

> **Note:** The uppermost track(s) may show very faint signal on the positive
> strand — this is minimal and can be ignored. Focus on the tracks with
> substantial signal, which correspond to the negative strand where APOL4 sits.

---
## Step 6 · Score the variant quantitatively

The plot gives us a visual picture, but we also want **numbers**. The
`score_variant` function:

1. Computes a **raw score** — the magnitude of the REF→ALT change for each
   gene–track combination
2. Converts this to a **quantile score** (0 to 1) — how extreme this variant's
   effect is compared to a background set of common benign variants

The quantile score is especially useful because it lets you compare across
different assay types (e.g. "is this variant more impactful on expression or
chromatin accessibility?") on a common scale.

We use the **recommended scorer** for RNA-seq, which the AlphaGenome team has
already optimised for this output type.

In [ ]:
# Get the recommended scorer configuration for RNA-seq
scorer = variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']

# Score the variant
# ⏱️ This makes an additional API call (~30 seconds)
variant_scores = dna_model.score_variant(
    interval=interval,
    variant=variant,
    variant_scorers=[scorer],
)

# Extract the first (and only) scorer's results
variant_scores = variant_scores[0]
print(f"✅ Scoring complete!")
print(f"   Scores matrix shape: {variant_scores.X.shape}")
print(f"   = {variant_scores.X.shape[0]} genes × {variant_scores.X.shape[1]} RNA-seq tracks")

### Understanding the output: AnnData

The scores come back as an **AnnData** object — a data structure widely used in
single-cell genomics. Think of it as a matrix with rich annotations:

| Component | What it contains | Analogy |
|-----------|-----------------|---------|
| `.X` | The score matrix (genes × tracks) | The spreadsheet values |
| `.obs` | Gene metadata (symbol, strand, coordinates) | Row labels |
| `.var` | Track metadata (assay, tissue, ontology) | Column labels |
| `.uns` | Extra info (interval, variant, scorer used) | Footnotes |

Let's peek at each:

In [ ]:
# What genes are in the interval?
print("Genes in the interval (first 5):")
print(variant_scores.obs.head())

In [ ]:
# What tracks were scored?
print(f"\nNumber of RNA-seq tracks: {variant_scores.X.shape[1]}")
print("\nTrack metadata (first 5):")
print(variant_scores.var.head())

### Flatten to a tidy table

The AnnData matrix is powerful but hard to browse. The `tidy_scores()` function
flattens everything into a single DataFrame where each row is one
gene–track–score combination.

It also adds the **quantile score** and optionally filters out tracks that don't
match the gene's strand (so we don't see misleading scores from the wrong
strand).

Let's look at the top results, sorted by the strongest effect:

In [ ]:
# Flatten scores into a tidy DataFrame
tidy = variant_scorers.tidy_scores(
    [variant_scores],
    match_gene_strand=True,  # Only keep tracks matching the gene's strand
)

# Sort by absolute raw score to find the most affected gene-tissue combinations
tidy_sorted = tidy.reindex(tidy['raw_score'].abs().sort_values(ascending=False).index)

print(f"Total gene–track combinations: {len(tidy_sorted):,}")
print(f"\nTop 10 most affected:")
tidy_sorted.head(10)

### 🔍 What to look for in the scores table

Key columns:
- **gene_symbol** — which gene is affected
- **ontology_name** — which tissue/cell type
- **raw_score** — direction and magnitude of the effect (negative = lower
  expression with ALT allele)
- **quantile_score** — how extreme this effect is (closer to 0 or 1 = more
  extreme; 0.5 = no effect)

You should see that:
- **APOL4** has the strongest effects (consistent with our plot)
- The effect is specifically in **colon** tracks
- The quantile scores are close to 0 or 1, indicating this is a genuinely
  impactful variant, not background noise

---
## Recap — What we did

In just a few lines of Python, we:

1. **Defined** a single nucleotide variant (chr22:36,201,698 A→C)
2. **Predicted** how it changes RNA-seq signal in colon tissue
3. **Visualised** the REF vs ALT predictions and saw both reduced expression
   and exon skipping in *APOL4*
4. **Scored** the variant quantitatively across all genes and tracks in the
   region
5. **Identified** the most affected gene–tissue combinations using quantile-
   calibrated scores

This is the core workflow for variant effect prediction with AlphaGenome. The
same approach works for any variant, any output type (ATAC, DNASE, ChIP-seq,
etc.), and any supported tissue.

---

### What else can AlphaGenome do?

| Capability | Function | Use case |
|------------|----------|----------|
| Predict from raw sequence | `predict_sequence()` | Synthetic/mutant sequences |
| Predict for a gene/region | `predict_interval()` | Explore a gene's regulatory landscape |
| In silico mutagenesis | `score_ism_variants()` | Find every important base in a regulatory element |
| Mouse predictions | `organism=Organism.MUS_MUSCULUS` | Cross-species analysis |

We'll explore several of these in the **AlphaGenome Explorer app** in Part 3!

---

*Full documentation: [alphagenomedocs.com](https://www.alphagenomedocs.com) ·
Tutorials: [alphagenomedocs.com/tutorials](https://www.alphagenomedocs.com/tutorials) ·
Paper: [Avsec, Latysheva, Cheng et al., Nature (2026)](https://www.nature.com/articles/s41586-025-10014-0)*